# Importing libraries

- Cosine Difference
- Vandermonde matrix

In [2]:
import numpy as np
from numpy.typing import NDArray
from typing import Annotated, Literal

Matrix2D = NDArray[np.number]
Vector = Annotated[NDArray[np.number], Literal["N"]]

# *1. Triangular system*

Given a positive integer $n > 2$, create the matrix $A \in \mathbf{R}^{n \times n}$ such that its diagonal elements are all 2; the upper triangular part is all $\frac{1}{2}$ and the lower triangular part is all $-\frac{1}{2}$.

For example, when $n = 4$:

$$
A = \begin{bmatrix}
2 & 0.5 & 0.5 & 0.5 \\
-0.5 & 2 & 0.5 & 0.5 \\
-0.5 & -0.5 & 2 & 0.5 \\
-0.5 & -0.5 & -0.5 & 2
\end{bmatrix}
$$

In [3]:
def triangular_matrix(n: int) -> Matrix2D:
    """
    A =
        2 0.5 0.5 0.5
        -0.5 2 0.5 0.5
        -0.5 -0.5 2 0.5
        -0.5 -0.5 -0.5 2
    """
    
    # Note : we'll use np.diag() for diagonal line and
    ## we'll use np.triu/l() for upper/lower triangular
    ## but , np.tril/u(matrix , k) -> for k = 0 will include diagonal
    ## so in this case we'll use k = +-1 for not including value in diagonal line
        
    # Step 1 : Define ndarray with nxn matrix 
    new_matrix = np.zeros((n,n)) # shape nxn

    # Step 2 : Define matrix then concat all matrix 
    diag_matrix = 2 * np.eye(n) # diagonal value = 2
    
    ## Note that upper & lower matrix will generate nxn matrix 
    upper_tri_matrix = np.triu(0.5 * np.ones((n,n)) , k = 1) # upper triangular value = 0.5
    lower_tri_matrix = np.tril(-0.5 * np.ones((n,n)) , k = -1) # lower triangular value = -0.5
    
    # Step 3 : Add all matrix together
    new_matrix += (diag_matrix + upper_tri_matrix + lower_tri_matrix)
    # Return the resulting matrix
    return new_matrix

In [4]:
n = 3
A = triangular_matrix(3)
SOL = np.array([[2, 0.5, 0.5], [-0.5, 2, 0.5], [-0.5, -0.5, 2]])
if np.allclose(A, SOL):
    print("test passed")
else:
    print("test failed")
    print(f"{A=}")

test passed


# *2. Cosine of difference.*
Given a vector $x \in \mathbf{R}^n$ where $x = (x_1,x_2,\ldots,x_n)$. Create the matrix $A \in \mathbf{R}^{n \times n}$ whose elements are $a_{ij} = \cos(x_i - x_j)$.

In [7]:
def cosine_of_difference(x: Vector) -> Matrix2D:
    # This task should use broadcasting method to subtract matrix together
    
    # Step 1 : get horizontal & vertical array from x
    ## Note that shouldn't use Transpose bcz x is 1D array (making it to be 2d array for broadcasting)
    ## so using reshape is better solution 
    x_horizontal_i = x.copy().reshape(1,-1) # row vector -> 1xn 
    x_vertical_j = x.copy().reshape(-1,1) # column vector -> nx1
    
    # Step 2 : calculate cosine difference
    return np.cos(x_horizontal_i - x_vertical_j) # Return the resulting matrix

In [8]:
x = np.array([0, np.pi/2, np.pi])
A = cosine_of_difference(x)
SOL = np.array([[1, 0, -1], [0, 1, 0], [-1, 0, 1]])
if np.allclose(A, SOL):
    print("test passed")
else:
    print("test failed")
    print(f"{A=}")

test passed


# *3. Stochastic matrix.*
Given a positive integer $n > 3$, randomly generate a matrix $A \in \mathbf{R}^{n \times n}$ such that i) $a_{ij} \geq 0$ for all $i,j$, and ii) $\sum_{i=1}^n a_{ij} = 1$ for $j=1,2,\ldots,n$.

For example, when $n = 3$:
$$
A = \begin{pmatrix}
0.2 & 0.1 & 0.4 \\
0.5 & 0.6 & 0.3 \\
0.3 & 0.3 & 0.3
\end{pmatrix}
$$
(The answer is not unique, but need to satisfy the conditions.)

In [9]:
def stochastic_matrix(n: int) -> Matrix2D:
    ## Condition of stochastic matrix
    # 1. all member in matrix must be positive
    # 2. each column sum must be 1
    # 3. 80% must not be zero
    
    # Step 1 : generate randomed matrix nxn
    randomed_matrix = np.random.rand(n,n)
    
    # Step 2 : get summation of each column with np.sum
    column_summation = np.sum(randomed_matrix , axis = 0) # axis 0 = vertical
    
    # Step 3 : normalize value with column_summation
    normalized_matrix = randomed_matrix / column_summation

    # return normalized matrix    
    return normalized_matrix # Return the stochastic matrix 

In [10]:
try:
    for n in range(4, 16):
        A = stochastic_matrix(n)
        assert A.shape == (n, n), f"Shape of A is {A.shape} but expected (n, n)"
        assert np.allclose(A.sum(axis=0), np.ones(n)), "Not all columns sum to 1"
except AssertionError as e:
    print(f"Test failed on n = {n}")
    print(f"{A=}")
    print(e)
else:
    print("All tests passed")

All tests passed


# *4. Toeplitz matrix.*
Given a large positive integer $n > 3$, create a matrix $A \in \mathbf{R}^{n \times n}$ such that

$$
a_{ij} = \begin{cases} 2 , & i=j ,\\
 (-0.5)^{|i-j|}, & i \neq j
 \end{cases}
$$

For example, when $n = 4$:
$$
A = \begin{pmatrix}
2 & -0.5 & 0.25 & -0.125 \\
-0.5 & 2 & -0.5 & 0.25 \\
0.25 & -0.5 & 2 & -0.5 \\
-0.125 & 0.25 & -0.5 & 2
\end{pmatrix}
$$

In [11]:
def toeplitz_matrix(n: int) -> Matrix2D:
    ## This matrix is like Diagonal Triangular matrix but like stair
    
    ## Step 1 : create nxn matrix for diffrence of exponent
    ## define zeros array 
    exponent_difference = np.zeros((n,n))
    ## define array from range 1-n
    num_range_horizontal_i = np.arange(1,n+1,1).reshape(1,n)
    num_range_vertical_j = num_range_horizontal_i.copy().reshape(-1,1)
    ## broadcast then subtract
    exponent_difference = abs(num_range_horizontal_i - num_range_vertical_j)

    # Step 2 : define value from function of toeplitz
    ## exponent value first
    toeplitz = (-0.5) ** exponent_difference
    ## fill diagonal using np.fill_diagonal
    np.fill_diagonal(toeplitz , 2)
    
    ## return matrix
    return toeplitz

In [12]:
n = 5
A = toeplitz_matrix(5)
SOL = np.array([[2, -0.5, 0.25, -0.125, 0.0625], [-0.5, 2, -0.5, 0.25, -0.125], [0.25, -0.5, 2, -0.5, 0.25], [-0.125, 0.25, -0.5, 2, -0.5], [0.0625, -0.125, 0.25, -0.5, 2]])
if np.allclose(A, SOL):
    print("test passed")
else:
    print("test failed")
    print(f"{A=}")

test passed


# *5. Hankel matrix.*
Given an $n$-dimensional vector $x=(x_1,x_2,\ldots,x_n)$, create the matrix $A$ that has $x$ along the anti-diagonal. For example, for $n=3$

$$
A = \begin{bmatrix}
0 &0 & x_3 \\
0 & x_2 & 0 \\
x_1 & 0 & 0
\end{bmatrix}
$$

In [17]:
def hankel_matrix(x: Vector) -> Matrix2D:
    # This matrix is flip axis of diagonal from \ to /
    ## Easy way : using np.fliplr(np.diag(x)) can solve this problem
    
    # Step 0 : get array size from x (using np.shape) & reverse x 
    m_size = x.shape[0] # (row , col , ...) -> row
    x_reverse = x[::-1]
    
    # Step 1 : define zero array
    hankel_mat = np.zeros((m_size , m_size)) # nxn matrix
    
    # Step 2 : slicing member from matrix then replace in hankel_mat
    ## Note : slicing matrix with array[ row_slice , col_slice ]
    hankel_mat[ np.arange(0 , m_size , 1) , np.arange(m_size-1 , -1 , -1) ] = x_reverse  
    
    # return matrix 
    return hankel_mat # Return the resulting matrix

In [18]:
x = np.array([1, 2, 3])
A = hankel_matrix(x)
SOL = np.array([[0, 0, 3], [0, 2, 0], [1, 0, 0]])
if np.allclose(A, SOL):
    print("test passed")
else:
    print("test failed")
    print(f"{A=}")

test passed


# *6. Vandermonde matrix.*
Let $t = (t_1,t_2,\ldots,t_n) \in \mathbf{R}^n$. Create the matrix $V \in \mathbf{R}^{n \times n}$ such that each column of $V$ is a geometric series of $t$.
$$
V = \begin{bmatrix}
1 & t_1 & t_1^2 & \cdots & t_1^{n-1} \\
1 & t_2 & t_2^t & \cdots & t_2^{n-1} \\
\vdots & & & \vdots & \vdots \\
1 & t_n & t_n^t & \cdots & t_n^{n-1} \\
\end{bmatrix}
$$


In [19]:
def vandermonde_matrix(t: Vector) -> Matrix2D:
    ## This matrix is square from matrix t as row
    
    ## Step 1 : Define nxn matrix as result
    n = t.shape[0] # n = matrix size = #row = #col
    
    ## using np.tile to copy 1 row to be same as first row in exponent_matrix
    
    ## Step 2 : define base matrix for doing operation later
    ## transpose / reshape to vertical (nx1) then tile horizontally 
    base_matrix = np.tile(t.reshape(-1,1) , (1 , n) )
        # tile matrix with vertical (#shape time) and horizontal (1 time)
    
    ## Step 3 : define exponent base (tile vertically)
    exponent_matrix = np.tile(np.arange(0 , n , 1) , (n , 1))
    
    ## return operated matrix
    vandermonde = base_matrix ** exponent_matrix # element-wise operation
    return vandermonde # Return the resulting matrix

In [20]:
x = np.array([1, 2, 3])
A = vandermonde_matrix(x)
SOL = np.array([[1, 1, 1], [1, 2, 4], [1, 3, 9]])
if np.allclose(A, SOL):
    print("test passed")
else:
    print("test failed")

test passed


# *7. Product of all possible differences.*
Let $t = (t_1,t_2,\ldots,t_n) \in \mathbf{R}^n$ and all elements are distinct, i.e., $t_i \neq t_j$ for all $i,j$. Calculate

$$
d = \prod_{i \neq j, i=1,2,\ldots,n} (t_i-t_j)
$$
For example, for $n=4$, the result is

$$
d = (t_1-t_2)(t_1-t_3)(t_1-t_4)(t_2-t_3)(t_2-t_4)(t_3-t_4)
$$

In [ ]:
def product_of_all_possible_differences(t: Vector) -> np.floating:
    return ...

In [ ]:
x = np.array([7, 2, 5])
d = product_of_all_possible_differences(x)
SOL = -30
if np.allclose(d, SOL):
    print("test passed")
else:
    print("test failed")
    print(f"{d=}, expected {SOL}")

# *8. Permutation matrix*

Given a positive integer $n$, create an $n \times n$ matrix $P$ such that each column and each row of $P$ contains only one non-zero entry as $1$. The identity  matrix is a trivial example. For example, $n=3$, it can be

$$
P_1 = \begin{bmatrix}
0 & 1 & 0 \\
1 & 0 & 0 \\
0 & 0 & 1
\end{bmatrix} \qquad
P_2 = \begin{bmatrix}
0 & 0 & 1\\
1 & 0 & 0 \\
0 & 1 & 0
\end{bmatrix}
$$

In [ ]:
def permutation_matrix(n: int) -> Matrix2D:
    return ...

print(permutation_matrix(3))

In [ ]:
try:
    for n in range(2, 16):
        P = permutation_matrix(n)
        assert P.shape == (n, n), f"Shape of P is {P.shape} but expected (n, n)"
        assert np.all(np.sum(P == 1, axis=0) == 1), "Some columns don't have exactly one 1"
        assert np.all(np.sum(P == 1, axis=1) == 1), "Some rows don't have exactly one 1"

except AssertionError as e:
    print(f"Test failed on n = {n}")
    print(e)
else:
    print("All tests passed")

# *9. Repeat matrix*

Given a positive integer $n$, create a matrix $A$ of size $(n+1) \times (n+1)$ where each row has identical entries and the values across the rows progress as the arithmetic sequence $2k+1$ for $k=0,1,\ldots,n$. For example, for $n=3$,

$$
A = \begin{bmatrix}
1 & 1 & 1 & 1 \\
3 & 3 & 3 & 3 \\
5 & 5 & 5 & 5 \\
7 & 7 & 7 & 7
\end{bmatrix}
$$

In [ ]:
def repeat_matrix(n: int) -> Matrix2D:
    return ...

In [ ]:
n = 3
A = repeat_matrix(3)
SOL = np.array([[1, 1, 1, 1], [3, 3, 3, 3], [5, 5, 5, 5], [7, 7, 7, 7]])
if np.allclose(A, SOL):
    print("test passed")
else:
    print("test failed")